# MDP and Bellman equations: lab notebook

**Machine Learning (5CS4-05), Semi-supervised and reinforcement learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np

### 1. The classic 3 x 4 grid world (reward is received on ENTERING a cell)

In [ ]:
# 1. The classic 3 x 4 grid world (reward is received on ENTERING a cell)
rows = ["...G", ".#.X", "S..."]
R, C = len(rows), len(rows[0])
STEP, SLIP = -0.04, 0.2
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # up, right, down, left
ARROW = "^>v<"
cell = lambda s: rows[s // C][s % C]
terminal = lambda s: cell(s) in "GX"
wall = lambda s: cell(s) == "#"
reward = lambda s: {"G": 1.0, "X": -1.0}.get(cell(s), STEP)

def move(s, a):
    r, c = s // C + ACTIONS[a][0], s % C + ACTIONS[a][1]
    return s if not (0 <= r < R and 0 <= c < C) or rows[r][c] == "#" else r * C + c

def transitions(s, a):
    """list of (probability, next_state, reward)"""
    if terminal(s) or wall(s):
        return [(1.0, s, 0.0)]
    out = [(1 - SLIP, a), (SLIP / 2, (a + 1) % 4), (SLIP / 2, (a + 3) % 4)]
    return [(p, move(s, b), reward(move(s, b))) for p, b in out]

def q_value(s, a, V, gamma):
    return sum(p * (r + gamma * V[s2]) for p, s2, r in transitions(s, a))

def show(V):
    for i in range(R):
        print("  ".join(f"{V[i * C + j]:6.3f}" if not wall(i * C + j) else "  wall" for j in range(C)))

### 2. Iterative policy evaluation of the uniform random policy (Bellman expectation equation)

In [ ]:
# 2. Iterative policy evaluation of the uniform random policy (Bellman expectation equation)
gamma = 0.9
nS = R * C
V = np.zeros(nS)
for sweep in range(1, 1000):
    newV = V.copy()
    for s in range(nS):
        if terminal(s) or wall(s):
            continue
        newV[s] = sum(0.25 * q_value(s, a, V, gamma) for a in range(4))
    delta = np.abs(newV - V).max()
    V = newV
    if sweep in (1, 2, 5) or delta < 1e-8:
        print(f"sweep {sweep:3d}  max change = {delta:.6f}")
    if delta < 1e-8:
        break
print("V for the uniform random policy:")
show(V)

### 3. Check the Bellman expectation equation in the start cell (2, 0)

In [ ]:
# 3. Check the Bellman expectation equation in the start cell (2, 0)
s = 2 * C
qs = [q_value(s, a, V, gamma) for a in range(4)]
print("Q(start, up/right/down/left):", np.round(qs, 3), " V(start) = mean(Q) =", round(np.mean(qs), 3))

### 4. The equation as a linear system: V = r_pi + gamma P_pi V

In [ ]:
# 4. The equation as a linear system: V = r_pi + gamma P_pi V
P = np.zeros((nS, nS)); r_pi = np.zeros(nS)
for s in range(nS):
    if terminal(s) or wall(s):
        continue
    for a in range(4):
        for p, s2, r in transitions(s, a):
            P[s, s2] += 0.25 * p; r_pi[s] += 0.25 * p * r
V_exact = np.linalg.solve(np.eye(nS) - gamma * P, r_pi)
print("solved exactly with linear algebra, max difference:", np.abs(V_exact - V).max())

## Expected output

Running all cells above should print:

```
sweep   1  max change = 0.280000
sweep   2  max change = 0.144000
sweep   5  max change = 0.044338
sweep 110  max change = 0.000000
V for the uniform random policy:
-0.275  -0.144   0.100   0.000
-0.350    wall  -0.488   0.000
-0.403  -0.458  -0.538  -0.729
Q(start, up/right/down/left): [-0.37  -0.437 -0.408 -0.398]  V(start) = mean(Q) = -0.403
solved exactly with linear algebra, max difference: 6.203449826047702e-08
```

## Practice exercises

Try these in the cells below.

1. Change gamma to 0.5 and to 0.99 in the script. How do the values near the goal and far from it change?
2. Compute V for a policy that always goes up. Which cells have a very negative value, and why?
3. Check by hand that the printed V satisfies the Bellman expectation equation in one cell.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
